# Heteroskedastik BNN: Aleatorik/Epistemik Belirsizlik + Chance Constraint + CVaR

Bu notebook, endüstri mühendisliği açısından sık görülen şu durumu ele alır:

> Talebin yalnızca ortalaması değil, **varyansı da koşullara göre değişiyor**.

Örneğin promosyon günlerinde veya aşırı sıcaklıklarda talep hem yükseliyor hem de daha oynak hale gelebilir. Sabit gürültü varsayımı bu yapıyı kaçırır.

Akış:

```text
heteroskedastik veri
      ↓
Pyro ile BNN
      ↓
μ_w(x) ve σ_w(x)
      ↓
Var(Y|x,D) = E_w[σ_w²(x)] + Var_w[μ_w(x)]
      ↓
posterior predictive senaryolar
      ↓
chance-constrained üretim
ve
CVaR-duyarlı üretim
```

Bu ayrıştırma **toplam varyans yasasına** dayanır. Buradaki aleatorik bileşen, posterior boyunca ortalama koşullu gözlem varyansıdır; epistemik bileşen ise koşullu ortalamanın posterior varyansıdır.


In [ ]:
# Gerekirse:
# %pip install torch pyro-ppl numpy pandas matplotlib pyomo highspy

import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

import pyro
import pyro.distributions as dist
from pyro.infer import SVI, Trace_ELBO, Predictive
from pyro.infer.autoguide import AutoDiagonalNormal
from pyro.nn import PyroModule, PyroSample

import pyomo.environ as pyo

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
pyro.set_rng_seed(SEED)
torch.set_default_dtype(torch.float32)


## 1. Heteroskedastik sentetik talep verisi

Gerçek sistemde `true_sigma` bilinmez; burada yalnızca yöntemin gerçekten değişen varyansı öğrenip öğrenemediğini görebilmek için sentetik olarak tanımlıyoruz.

- `promotion`: promosyon günü
- `temperature`: sıcaklık
- `weekend`: hafta sonu
- `trend`: zaman trendi

Koşullu ortalama:

\[
\mu(x)=E[Y\mid x]
\]

Koşullu standart sapma:

\[
\sigma(x)=\sqrt{Var(Y\mid x)}
\]

ve gözlem modeli:

\[
Y\mid x \sim \mathcal N(\mu(x),\sigma^2(x)).
\]


In [ ]:
n = 320
day = np.arange(n)

promotion = np.random.binomial(1, 0.25, n)
weekend = ((day % 7) >= 5).astype(float)
temperature = 22 + 9*np.sin(2*np.pi*day/90) + np.random.normal(0, 2.2, n)
season = np.sin(2*np.pi*day/30)
trend = day/(n-1)

true_mu = (
    105
    + 17*promotion
    + 9*weekend
    + 13*season
    + 0.45*temperature
    + 8*trend
)

true_sigma = (
    4.0
    + 5.5*promotion
    + 3.0*weekend
    + 0.22*np.abs(temperature - 22)
)

demand = true_mu + np.random.normal(0, true_sigma)

df = pd.DataFrame({
    "day": day,
    "promotion": promotion,
    "weekend": weekend,
    "temperature": temperature,
    "season": season,
    "trend": trend,
    "demand": demand,
    "true_mu": true_mu,
    "true_sigma": true_sigma,
})

df.head()


In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.scatter(day, demand, s=12, alpha=0.55, label="Gözlenen talep")
ax.plot(day, true_mu, linewidth=1.5, label="Gerçek koşullu ortalama")
ax.fill_between(
    day,
    true_mu - 2*true_sigma,
    true_mu + 2*true_sigma,
    alpha=0.18,
    label="Yaklaşık ±2σ bandı",
)
ax.set_xlabel("Gün")
ax.set_ylabel("Talep")
ax.legend()
plt.show()


## 2. Veriyi ölçekleme

BNN'lerde standardizasyon, hem optimizasyonu hem de prior seçimini kolaylaştırır.


In [ ]:
features = ["promotion", "weekend", "temperature", "season", "trend"]

X_np = df[features].to_numpy(np.float32)
y_np = df["demand"].to_numpy(np.float32)

X_mean = X_np.mean(axis=0, keepdims=True)
X_std = X_np.std(axis=0, keepdims=True) + 1e-6
y_mean = float(y_np.mean())
y_std = float(y_np.std() + 1e-6)

X = torch.tensor((X_np - X_mean) / X_std)
y = torch.tensor((y_np - y_mean) / y_std)

print("X shape:", X.shape, "y shape:", y.shape)


## 3. Heteroskedastik Bayesçi sinir ağı

Ağ iki fonksiyon öğreniyor:

\[
\mu_w(x)
\]

ve

\[
\sigma_w(x)>0.
\]

Ağırlıklar rassal değişken olduğu için `w` üzerinde posterior belirsizlik vardır. Gözlem modeli:

\[
Y\mid x,w \sim \mathcal N(\mu_w(x),\sigma_w^2(x)).
\]

`softplus` kullanımı \(\sigma_w(x)>0\) koşulunu sağlar.


In [ ]:
class HeteroscedasticBNN(PyroModule):
    def __init__(self, in_features, hidden=20):
        super().__init__()

        self.hidden = PyroModule[nn.Linear](in_features, hidden)
        self.hidden.weight = PyroSample(
            dist.Normal(0.0, 0.8).expand([hidden, in_features]).to_event(2)
        )
        self.hidden.bias = PyroSample(
            dist.Normal(0.0, 0.8).expand([hidden]).to_event(1)
        )

        self.mean_head = PyroModule[nn.Linear](hidden, 1)
        self.mean_head.weight = PyroSample(
            dist.Normal(0.0, 0.8).expand([1, hidden]).to_event(2)
        )
        self.mean_head.bias = PyroSample(
            dist.Normal(0.0, 0.8).expand([1]).to_event(1)
        )

        self.scale_head = PyroModule[nn.Linear](hidden, 1)
        self.scale_head.weight = PyroSample(
            dist.Normal(0.0, 0.5).expand([1, hidden]).to_event(2)
        )
        self.scale_head.bias = PyroSample(
            dist.Normal(-1.0, 0.5).expand([1]).to_event(1)
        )

    def forward(self, x, y=None):
        h = torch.tanh(self.hidden(x))

        mu = self.mean_head(h).squeeze(-1)
        raw_scale = self.scale_head(h).squeeze(-1)
        sigma_x = 0.03 + F.softplus(raw_scale)

        pyro.deterministic("mu", mu)
        pyro.deterministic("sigma_x", sigma_x)

        with pyro.plate("data", x.shape[0]):
            pyro.sample("obs", dist.Normal(mu, sigma_x), obs=y)

        return mu


## 4. Variational inference ile posterior yaklaşımı

`AutoDiagonalNormal`, ağırlık posteriorunu mean-field Gaussian ile yaklaşıklar.

Bu hızlı ve kullanışlıdır, fakat tam posterior değildir. Risk-kritik uygulamada HMC/NUTS, daha zengin variational posterior veya ensemble gibi yöntemlerle hassasiyet analizi yapmak gerekir.


In [ ]:
pyro.clear_param_store()

model = HeteroscedasticBNN(X.shape[1], hidden=20)
guide = AutoDiagonalNormal(model)

svi = SVI(
    model,
    guide,
    pyro.optim.Adam({"lr": 0.01}),
    loss=Trace_ELBO(),
)

losses = []
for step in range(3500):
    loss = svi.step(X, y) / len(y)
    losses.append(loss)

    if (step + 1) % 700 == 0:
        print(f"Adım {step+1}: ELBO/gözlem = {loss:.4f}")

plt.figure(figsize=(8, 3))
plt.plot(losses)
plt.xlabel("SVI adımı")
plt.ylabel("ELBO / gözlem")
plt.show()


## 5. Aleatorik ve epistemik belirsizliği ayırma

Toplam varyans yasası:

\[
Var(Y\mid x,\mathcal D)
=
E_{w\mid\mathcal D}\left[Var(Y\mid x,w)\right]
+
Var_{w\mid\mathcal D}\left(E[Y\mid x,w]\right).
\]

Bu modelde:

\[
E[Y\mid x,w]=\mu_w(x)
\]

ve

\[
Var(Y\mid x,w)=\sigma_w^2(x).
\]

Dolayısıyla Monte Carlo yaklaşımı:

\[
\widehat{V}_{alea}
=
\frac{1}{S}\sum_s \sigma_{w_s}^2(x)
\]

\[
\widehat{V}_{epi}
=
Var_s(\mu_{w_s}(x)).
\]

Önemli nüans: `sigma_x` fonksiyonunun parametreleri de Bayesçidir. Burada “aleatorik” diye raporlanan büyüklük, posterior boyunca **beklenen koşullu veri varyansıdır**; `sigma(x)` fonksiyonunun kendisi üzerindeki model belirsizliğini ayrıca üçüncü bir terime ayırmıyoruz.


In [ ]:
def posterior_at(raw_x, num_samples=3000):
    raw_x = np.asarray(raw_x, dtype=np.float32).reshape(1, -1)
    x_scaled = torch.tensor((raw_x - X_mean) / X_std)

    predictive = Predictive(
        model,
        guide=guide,
        num_samples=num_samples,
        return_sites=("obs", "mu", "sigma_x"),
    )
    out = predictive(x_scaled)

    obs = out["obs"].detach().cpu().numpy().reshape(-1) * y_std + y_mean
    mu = out["mu"].detach().cpu().numpy().reshape(-1) * y_std + y_mean
    sigma = out["sigma_x"].detach().cpu().numpy().reshape(-1) * y_std

    return obs, mu, sigma

# [promotion, weekend, temperature, season, trend]
low_noise_x = [0, 0, 22, 0.2, 1.02]
high_noise_x = [1, 1, 34, 0.2, 1.02]

low_obs, low_mu, low_sigma = posterior_at(low_noise_x)
high_obs, high_mu, high_sigma = posterior_at(high_noise_x)

def uncertainty_decomposition(obs, mu, sigma):
    aleatoric_var = float(np.mean(sigma**2))
    epistemic_var = float(np.var(mu, ddof=1))
    total_from_decomposition = aleatoric_var + epistemic_var
    empirical_predictive_var = float(np.var(obs, ddof=1))

    return {
        "aleatorik_std": np.sqrt(aleatoric_var),
        "epistemik_std": np.sqrt(epistemic_var),
        "toplam_std_ayristirma": np.sqrt(total_from_decomposition),
        "predictive_std_empirik": np.sqrt(empirical_predictive_var),
        "aleatorik_varyans_payi": aleatoric_var / total_from_decomposition,
    }

summary = pd.DataFrame(
    [
        {"Nokta": "Düşük gürültü", **uncertainty_decomposition(low_obs, low_mu, low_sigma)},
        {"Nokta": "Yüksek gürültü", **uncertainty_decomposition(high_obs, high_mu, high_sigma)},
    ]
)

summary


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(low_obs, bins=45, density=True, alpha=0.45, label="Düşük gürültü noktası")
ax.hist(high_obs, bins=45, density=True, alpha=0.45, label="Yüksek gürültü noktası")
ax.set_xlabel("Posterior predictive talep")
ax.set_ylabel("Yoğunluk")
ax.legend()
plt.show()


## 6. Chance-constrained üretim kararı

Karar değişkeni \(q\): üretilecek miktar.

İstenen hizmet seviyesi:

\[
P(D\le q)\ge 0.95.
\]

Posterior predictive senaryolarla bunun örneklem yaklaşımı:

\[
q + M z_s \ge D_s,
\]

\[
\sum_s z_s \le \lfloor \alpha S \rfloor,
\]

\[
z_s\in\{0,1\}.
\]

Bu, posterior senaryolar üzerindeki **ampirik chance constraint**'tir. Sonlu örneklem için otomatik olarak dağılımdan bağımsız güvence verdiği iddia edilmez; gerçek sistemde scenario-approach örneklem büyüklüğü teorisi veya out-of-sample doğrulama gerekir.


In [ ]:
rng = np.random.default_rng(SEED)

S = 500
scenarios_full = rng.choice(high_obs, size=S, replace=False)
scenarios_mean_only = rng.choice(high_mu, size=S, replace=False)

def solve_empirical_chance_constraint(scenarios, service_level=0.95):
    scenarios = np.asarray(scenarios, dtype=float)
    S = len(scenarios)
    alpha = 1.0 - service_level

    q_max = max(260.0, float(np.max(scenarios) + 10.0))
    big_m = float(np.max(scenarios) + 50.0)

    m = pyo.ConcreteModel()
    m.S = pyo.RangeSet(0, S - 1)

    m.q = pyo.Var(bounds=(0.0, q_max))
    m.z = pyo.Var(m.S, domain=pyo.Binary)

    D = {s: float(scenarios[s]) for s in range(S)}

    m.cover = pyo.Constraint(
        m.S,
        rule=lambda M, s: M.q + big_m * M.z[s] >= D[s]
    )

    max_violations = int(np.floor(alpha * S + 1e-9))
    m.service = pyo.Constraint(
        expr=sum(m.z[s] for s in m.S) <= max_violations
    )

    m.obj = pyo.Objective(expr=m.q, sense=pyo.minimize)

    result = pyo.SolverFactory("appsi_highs").solve(m)

    return {
        "q": pyo.value(m.q),
        "termination": str(result.solver.termination_condition),
        "violations": sum(round(pyo.value(m.z[s])) for s in m.S),
        "allowed": max_violations,
    }

chance_full = solve_empirical_chance_constraint(scenarios_full, service_level=0.95)
chance_mean = solve_empirical_chance_constraint(scenarios_mean_only, service_level=0.95)

pd.DataFrame([
    {"Model": "Tam posterior predictive (aleatorik + epistemik)", **chance_full},
    {"Model": "Yalnız posterior ortalama fonksiyonu", **chance_mean},
])


Yalnızca posterior ortalama fonksiyonunun örneklerini kullanmak, gözlem gürültüsünü dışarıda bırakır. Heteroskedastik sistemde bu, özellikle riskli koşullarda gerekli kapasiteyi sistematik olarak düşük gösterebilir.


## 7. CVaR-duyarlı üretim planlama

Senaryo maliyeti:

\[
L_s(q)
=
c_p q
+
c_h(q-D_s)^+
+
c_u(D_s-q)^+.
\]

CVaR için Rockafellar–Uryasev gösterimi:

\[
CVaR_\beta(L)
=
\min_{\eta}
\left[
\eta+
\frac{1}{1-\beta}
E[(L-\eta)^+]
\right].
\]

Senaryo yaklaşımında:

\[
\eta+
\frac{1}{(1-\beta)S}
\sum_s u_s
\]

minimize edilir ve

\[
u_s\ge L_s(q)-\eta,\quad u_s\ge0.
\]


In [ ]:
production_cost = 1.0
holding_cost = 0.8
shortage_cost = 8.0
q_upper = 260.0

def solve_expected_cost(scenarios):
    scenarios = np.asarray(scenarios, dtype=float)
    S = len(scenarios)

    m = pyo.ConcreteModel()
    m.S = pyo.RangeSet(0, S - 1)
    m.q = pyo.Var(bounds=(0, q_upper))
    m.excess = pyo.Var(m.S, domain=pyo.NonNegativeReals)
    m.shortage = pyo.Var(m.S, domain=pyo.NonNegativeReals)

    D = {s: float(scenarios[s]) for s in range(S)}

    m.excess_con = pyo.Constraint(
        m.S, rule=lambda M, s: M.excess[s] >= M.q - D[s]
    )
    m.shortage_con = pyo.Constraint(
        m.S, rule=lambda M, s: M.shortage[s] >= D[s] - M.q
    )

    m.obj = pyo.Objective(
        expr=production_cost*m.q
        + (1/S) * sum(
            holding_cost*m.excess[s] + shortage_cost*m.shortage[s]
            for s in m.S
        ),
        sense=pyo.minimize,
    )

    pyo.SolverFactory("appsi_highs").solve(m)
    return float(pyo.value(m.q))

def solve_cvar(scenarios, beta=0.95):
    scenarios = np.asarray(scenarios, dtype=float)
    S = len(scenarios)

    m = pyo.ConcreteModel()
    m.S = pyo.RangeSet(0, S - 1)

    m.q = pyo.Var(bounds=(0, q_upper))
    m.excess = pyo.Var(m.S, domain=pyo.NonNegativeReals)
    m.shortage = pyo.Var(m.S, domain=pyo.NonNegativeReals)

    m.eta = pyo.Var()
    m.u = pyo.Var(m.S, domain=pyo.NonNegativeReals)

    D = {s: float(scenarios[s]) for s in range(S)}

    m.excess_con = pyo.Constraint(
        m.S, rule=lambda M, s: M.excess[s] >= M.q - D[s]
    )
    m.shortage_con = pyo.Constraint(
        m.S, rule=lambda M, s: M.shortage[s] >= D[s] - M.q
    )

    def loss_expr(M, s):
        return (
            production_cost*M.q
            + holding_cost*M.excess[s]
            + shortage_cost*M.shortage[s]
        )

    m.tail = pyo.Constraint(
        m.S,
        rule=lambda M, s: M.u[s] >= loss_expr(M, s) - M.eta
    )

    m.obj = pyo.Objective(
        expr=m.eta
        + (1 / ((1 - beta) * S)) * sum(m.u[s] for s in m.S),
        sense=pyo.minimize,
    )

    pyo.SolverFactory("appsi_highs").solve(m)

    return float(pyo.value(m.q)), float(pyo.value(m.obj))

q_expected = solve_expected_cost(scenarios_full)
q_cvar, model_cvar = solve_cvar(scenarios_full, beta=0.95)

print("Beklenen maliyet optimum q:", round(q_expected, 2))
print("CVaR95 optimum q:", round(q_cvar, 2))
print("Model CVaR95 değeri:", round(model_cvar, 2))


## 8. Kararları out-of-sample posterior predictive örneklerde karşılaştırma

Aynı senaryoları hem optimizasyonda hem değerlendirmede kullanmak iyimser sonuç verebilir. Bu nedenle ayrı posterior predictive örnekleri kullanıyoruz.


In [ ]:
eval_obs, _, _ = posterior_at(high_noise_x, num_samples=6000)

def realized_cost(q, d):
    return (
        production_cost*q
        + holding_cost*max(q-d, 0.0)
        + shortage_cost*max(d-q, 0.0)
    )

def empirical_cvar(costs, beta=0.95):
    costs = np.asarray(costs)
    eta = np.quantile(costs, beta)
    return float(eta + np.mean(np.maximum(costs - eta, 0.0)) / (1-beta))

decisions = {
    "Beklenen maliyet": q_expected,
    "CVaR95": q_cvar,
    "Chance constraint %95": chance_full["q"],
    "Yalnız posterior ortalaması ile chance": chance_mean["q"],
}

rows = []
for name, q in decisions.items():
    costs = np.array([realized_cost(q, d) for d in eval_obs])
    rows.append({
        "Karar": name,
        "q": q,
        "Beklenen maliyet": costs.mean(),
        "CVaR95": empirical_cvar(costs, 0.95),
        "Stokout olasılığı": np.mean(eval_obs > q),
    })

comparison = pd.DataFrame(rows).sort_values("CVaR95")
comparison


## 9. Ne kanıtlandı, ne kanıtlanmadı?

Bu notebookta kullanılan yapıların matematiksel temeli standarttır:

1. **Heteroskedastik regresyon:** \(Y|x,w\sim N(\mu_w(x),\sigma_w^2(x))\)
2. **Bayesçi ağırlık belirsizliği:** \(w\sim p(w)\), veri sonrası posterior yaklaşımı
3. **Toplam varyans yasası:** aleatorik/epistemik ayrıştırmanın temelidir
4. **Chance constraints:** OR'da klasik olasılıksal kısıt sınıfıdır
5. **CVaR:** kuyruk riskini ölçen standart risk ölçüsüdür
6. **Rockafellar–Uryasev reformülasyonu:** CVaR'ı optimizasyona taşımak için klasik formülasyondur

Ancak aşağıdakiler **yaklaşım / modelleme tercihi**dir:

- mean-field variational posterior gerçek posteriorun yaklaşık halidir,
- Normal likelihood gerçek talep dağılımına uymayabilir,
- posterior predictive senaryolar gerçek dünyanın eksiksiz dağılımı değildir,
- ampirik chance constraint tek başına sonlu örneklem güvence teoremi değildir,
- BNN iyi kalibre edilmemişse CVaR ve hizmet seviyesi kararları da hatalı olabilir.

Bu nedenle gerçek endüstriyel kullanımda model kalibrasyonu ve kararların out-of-sample doğrulanması zorunludur.


## 10. Gerçek IE/OR projelerine genişletme

Aynı yapı şu belirsiz parametrelere uygulanabilir:

| BNN çıktısı | Heteroskedastisite örneği | Optimizasyon |
|---|---|---|
| Talep | promosyonda varyans artıyor | stok / üretim |
| İşlem süresi | ürün tipine göre varyans değişiyor | çizelgeleme |
| Lead time | tedarikçiye / sezona göre oynaklık | tedarik zinciri |
| RUL / arıza zamanı | yaşlı ekipmanda dağılım genişliyor | bakım |
| Taşıma süresi | saat ve güzergâha göre varyans | VRP / filo |
| Kalite | proses bölgesine göre ölçüm gürültüsü | proses optimizasyonu |

Araştırma karşılaştırması için en az şu baseline'lar önerilir:

- homoskedastik BNN,
- heteroskedastik BNN,
- deep ensemble,
- Gaussian Process (veri boyutu uygunsa),
- deterministik NN + residual model.

Değerlendirmede yalnızca RMSE değil:

- NLL,
- interval coverage,
- CRPS,
- calibration,
- beklenen operasyonel maliyet,
- CVaR,
- constraint violation rate

raporlanmalıdır.
